In [1]:
import pandas as pd


In [2]:
# from datasets import load_dataset

# dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_All_Beauty", trust_remote_code=True)
# data = dataset['full'].to_pandas()


# data = pd.read_csv("datasets/datafiniti/consumer-reviews-of-amazon-products/versions/5/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv")
# print(data.head())

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your sessi

README.md: 0.00B [00:00, ?B/s]

Amazon-Reviews-2023.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

data = pd.read_json("datasets/All_Beauty.jsonl", lines=True)

In [7]:
data.drop(columns=['images', 'asin', 'parent_asin', 'user_id','timestamp',], inplace=True)
data = data.dropna()
missing_counts = data.isna().sum()
print(missing_counts)

rating               0
title                0
text                 0
helpful_vote         0
verified_purchase    0
dtype: int64


In [8]:
ratings_map = {5:2,4:2,3:1,2:0,1:0}
data['rating_sentiment'] = data['rating'].map(ratings_map)
print(data.head())
# data.groupby('rating_sentiment')['reviews.doRecommend'].mean().plot(kind='bar')


   rating                                      title  \
0       5  Such a lovely scent but not overpowering.   
1       4     Works great but smells a little weird.   
2       5                                       Yes!   
3       1                          Synthetic feeling   
4       5                                         A+   

                                                text  helpful_vote  \
0  This spray is really nice. It smells really go...             0   
1  This product does what I need it to do, I just...             1   
2                          Smells good, feels great!             2   
3                                     Felt synthetic             0   
4                                            Love it             0   

   verified_purchase  rating_sentiment  
0               True                 2  
1               True                 2  
2               True                 2  
3               True                 0  
4               True                 

In [11]:
print((data['rating_sentiment'] == 2).sum())
print((data['rating_sentiment'] == 1).sum())
print((data['rating_sentiment'] == 0).sum())

min = min([(data['rating_sentiment'] == 2).sum(), (data['rating_sentiment'] == 1).sum(), (data['rating_sentiment'] == 1).sum()])
print(min)

500107
56307
145114
56307


In [12]:
df_pos = data[data['rating_sentiment'] == 2].sample(n=min, random_state=42) # Reduce 942 to min
df_neu = data[data['rating_sentiment'] == 1].sample(n=min, random_state=42)
df_neg = data[data['rating_sentiment'] == 0].sample(n=min, random_state=42)

df_balanced = pd.concat([df_pos, df_neu, df_neg])

In [13]:
print((df_balanced['rating_sentiment'] == 2).sum())
print((df_balanced['rating_sentiment'] == 1).sum())
print((df_balanced['rating_sentiment'] == 0).sum())

56307
56307
56307


In [14]:
from sklearn.model_selection import train_test_split
X=df_balanced["text"]
y=df_balanced["rating_sentiment"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (135136,)
Testing set size: (33785,)
Training output size: (135136,)
Testing output size: (33785,)


In [15]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score
import numpy as np

# 1. Load Tokenizer and Model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
# num_labels=2 for binary classification (Real vs Fake)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=3)


train_df = pd.DataFrame({"text": X_train, "label": y_train})
val_df = pd.DataFrame({"text": X_val, "label": y_val})
# 2. Convert your DataFrame to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

print(train_dataset[0])

# 3. Tokenization Function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print(tokenized_train[0])

# 4. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,              # 3 epochs is usually enough for fine-tuning
    eval_strategy='epoch',      # Evaluate after each epoch
    learning_rate=2e-5,              # Crucial: use a very small learning rate for fine-tuning
    weight_decay=0.01,
)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}
# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)




tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'text': 'Awesome price', 'label': 2}


Map:   0%|          | 0/135136 [00:00<?, ? examples/s]

Map:   0%|          | 0/33785 [00:00<?, ? examples/s]

{'text': 'Awesome price', 'label': 2, 'input_ids': [101, 12476, 3976, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [16]:
# 6. Start Fine-Tuning
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.534098,0.532881,0.778541
2,0.437887,0.536166,0.788545
3,0.395034,0.620728,0.786503


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50676, training_loss=0.4720860750073556, metrics={'train_runtime': 7668.3119, 'train_samples_per_second': 52.868, 'train_steps_per_second': 6.608, 'total_flos': 1.3426075219304448e+16, 'train_loss': 0.4720860750073556, 'epoch': 3.0})

In [17]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.6207280158996582, 'eval_accuracy': 0.7865028858961077, 'eval_runtime': 130.2579, 'eval_samples_per_second': 259.37, 'eval_steps_per_second': 32.428, 'epoch': 3.0}


In [1]:
# Get raw predictions
predictions = trainer.predict(tokenized_val)

# Convert raw numbers (logits) to class labels (0, 1, or 2)
predicted_labels = np.argmax(predictions.predictions, axis=1)

# # Compare them to the real labels
# true_labels = predictions.label_ids


# Your code here :
from sklearn.metrics import accuracy_score
print(accuracy_score(y_testset, y_test_pred))
print(accuracy_score(y_trainset, y_train_pred))

NameError: name 'trainer' is not defined

In [2]:
from transformers import pipeline

# Create a classification pipeline using your trained model and tokenizer
# If you didn't save it yet, use trainer.model and trainer.tokenizer
my_test_pipeline = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tokenizer)

# Try a positive review
print(my_test_pipeline("This tablet is amazing, the battery lasts forever!"))

# Try a negative review
print(my_test_pipeline("I hate this product, it broke after two days."))

NameError: name 'trainer' is not defined

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)

# best_row    = ens_df.iloc[0]
# best_name   = best_row['Config']
# w42, w0, wsvc, wlr = best_row['weights']
# total_w     = w42 + w0 + wsvc + wlr

# best_probs  = (w42  * probs_s42 +
#                w0   * probs_s0  +
#                wsvc * probs_svc +
#                wlr  * probs_lr) / total_w
# best_preds  = (best_probs[:, 1] >= 0.5).astype(int)

final_acc = accuracy_score(val_df['label'], predicted_labels)
# final_auc = roc_auc_score(val_df['label'], best_probs[:, 1])

# print(f'Best config     : {best_name}')
# print(f'Weights         : seed42={w42}  seed0={w0}  svc={wsvc}  lr={wlr}')
print(f'Final Accuracy  : {final_acc:.4f}  ({final_acc*100:.2f}%)')
# print(f'ROC-AUC         : {final_auc:.4f}')
print()
print(classification_report(val_df['label'], predicted_labels))

print('\nConfusion Matrix:')
print(confusion_matrix(val_df['label'], predicted_labels))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(val_df['label'], predicted_labels),
    display_labels=['Negative(0)', 'Neutral (1)', "Positive(2)"]
).plot(ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold')

report = classification_report(val_df['label'], predicted_labels,
                               target_names=['Neg', 'Neu', 'Pos'], output_dict=True)

# Plotting precision, recall, and f1-score on the second axis
pd.DataFrame(report).iloc[:-1, :3].T.plot(kind='bar', ax=axes[1])
axes[1].set_title('Precision, Recall & F1-Score', fontweight='bold')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

# # Model comparison
# names = ['DistilBERT\nseed=42\n(model4)', 'DistilBERT\nseed=0',
#          'TF-IDF\nSVC', 'TF-IDF\nLogReg', f'Ensemble\n(best)']
# accs  = [acc_s42, acc_s0, acc_svc, acc_lr, final_acc]
# cols  = ['#e67e22', '#f39c12', '#3498db', '#2ecc71', '#e74c3c']

# bars = axes[1].bar(names, accs, color=cols, edgecolor='white', width=0.6)
# for bar, acc in zip(bars, accs):
#     axes[1].text(bar.get_x() + bar.get_width()/2,
#                  bar.get_height() + 0.001,
#                  f'{acc*100:.2f}%', ha='center', fontsize=9, fontweight='bold')

# axes[1].set_ylim(0.90, 1.005)
# axes[1].axhline(0.984, ls='--', color='orange', lw=1.2, label='model4 baseline (98.4%)')
# axes[1].axhline(0.990, ls='--', color='red',    lw=1.2, label='99% target')
# axes[1].set_ylabel('Validation Accuracy')
# axes[1].set_title('Model Comparison', fontweight='bold')
# axes[1].legend(fontsize=8)
# axes[1].grid(axis='y', alpha=0.3)

# plt.tight_layout()
# plt.savefig('model6_evaluation.png', dpi=150, bbox_inches='tight')
# plt.show()

The 3 Core IssuesExtreme Class Imbalance: You have 942 positive reviews but only 20 negative and 36 neutral. The model "learned" that if it just guesses "Positive" (2), it gets a 96% accuracy grade.Neutral (1) is a "Ghost": With an F1-score of 0.30, the model is failing on neutrals. Look at the matrix: 26 out of 36 neutrals were misclassified as positive. It hasn't learned the "middle ground."Positive Over-confidence: The recall for Positive is 0.99. It’s basically sucking all the data into the positive category because it doesn't have enough "Negative" examples to know what a complaint looks like.

In [ ]:
train with larger data set??
change params of current model??
downsample positive reviews?


lets try all 3 separately and then TokenClassificationArgumentHandler

EDA IN PC